# memrust — RAG with real embeddings

This notebook builds a complete retrieval pipeline on memrust:

1. embed documents with **sentence-transformers** (`all-MiniLM-L6-v2`)
2. store text + vectors in memrust (`remember_batch` — bulk ingest)
3. hybrid retrieval: semantic vectors + BM25 + entity graph, fused, explained
4. **RAG**: ground an LLM answer in recalled memories (OpenAI key optional)
5. memory lifecycle: TTL expiry, consolidation, snapshot/restore

memrust supports embeddings three ways: a built-in OpenAI-compatible client,
a native Gemini client, or **bring-your-own vectors** — which is what we use
here, embedding locally in the notebook and passing vectors explicitly.

In [ ]:
%pip install -q memrust sentence-transformers

In [ ]:
# Download the memrust server (one static Linux binary) and start it.
# Local machine? Build with `cargo build --release` and point BIN at
# target/release/memrust instead.
import os, subprocess, time, urllib.request

BIN = "./memrust-bin/memrust"
DATA_DIR = "./memory-rag"          # this notebook's own memory store

if not os.path.exists(BIN):
    os.makedirs("memrust-bin", exist_ok=True)
    url = ("https://github.com/AIAnytime/memrust/releases/download/"
           "v0.5.1/memrust-v0.5.1-x86_64-unknown-linux-musl.tar.gz")
    urllib.request.urlretrieve(url, "memrust-bin/memrust.tar.gz")
    subprocess.run(["tar", "xzf", "memrust.tar.gz"], cwd="memrust-bin", check=True)

# Re-running this cell (or another memrust notebook in the same runtime) can
# leave an old server holding port 7700 with a different data dir — replace it.
subprocess.run(["pkill", "-f", "memrust serve"], capture_output=True)
time.sleep(0.5)

server = subprocess.Popen(
    [BIN, "serve", "--data-dir", DATA_DIR,
     "--lifecycle-interval-secs", "0",     # we trigger lifecycle explicitly below
     "--consolidate-after-secs", "0"],     # so consolidation demos run immediately
    stdout=subprocess.DEVNULL, stderr=subprocess.DEVNULL,
)
for _ in range(40):
    try:
        urllib.request.urlopen("http://127.0.0.1:7700/health", timeout=1)
        break
    except Exception:
        time.sleep(0.5)
print("memrust is up on http://127.0.0.1:7700 (data dir: memory-rag)")

In [ ]:
from memrust import MemrustClient
from sentence_transformers import SentenceTransformer

memory = MemrustClient("http://127.0.0.1:7700")
model = SentenceTransformer("all-MiniLM-L6-v2")   # 384-dim, fast, free

def emb(text):
    return model.encode(text, normalize_embeddings=True).tolist()

print("embedding dim:", model.get_sentence_embedding_dimension())

## Ingest a knowledge base

A small fictional company KB. The **first stored vector fixes the collection's
dimension** (384 here) — from then on, recalls must supply a matching
`query_embedding`, and memrust rejects mismatched vectors instead of silently
returning garbage.

In [ ]:
docs = [
    ("Orion is our internal feature-flag service; it stores flags in Postgres", "semantic", ["infra"]),
    ("Orion flag evaluations are cached for 30 seconds at the edge", "semantic", ["infra"]),
    ("Incident INC-2201: Orion returned stale flags after the cache TTL change", "episodic", ["incident"]),
    ("Rollbacks are performed with helm rollback within five minutes", "procedural", ["runbook"]),
    ("The mobile team consumes Orion through the GraphQL gateway", "semantic", ["infra"]),
    ("Q3 goal: migrate Orion storage from Postgres to the new KV store", "semantic", ["planning"]),
    ("Customer Meridian Labs reported flag latency spikes on 2026-07-12", "episodic", ["customer"]),
    ("Meridian Labs is on the enterprise tier with a 99.9 percent SLA", "semantic", ["customer"]),
    ("On-call handoff happens Mondays at 10:00 UTC", "semantic", ["process"]),
    ("Postmortems are due within 48 hours of incident resolution", "procedural", ["process"]),
    ("The KV store beta showed p99 read latency of 2ms in load tests", "episodic", ["planning"]),
    ("GraphQL gateway timeouts are set to 3 seconds end to end", "semantic", ["infra"]),
]

texts = [d[0] for d in docs]
vectors = model.encode(texts, normalize_embeddings=True).tolist()

stored = memory.remember_batch([
    {"text": t, "kind": kind, "tags": tags, "embedding": vec, "session_id": "kb"}
    for (t, kind, tags), vec in zip(docs, vectors)
])
print(f"stored {len(stored)} memories")
memory.health()

## Hybrid retrieval

`retrieve()` embeds the query with the same model and recalls. Two things to
watch in the signal breakdown:

- a **paraphrased** query (zero keyword overlap) rides the vector signal
- an **exact identifier** rides BM25 — the case pure vector search loses

In [ ]:
def show(hits):
    """Pretty-print recall hits with the per-signal score breakdown."""
    for h in hits:
        s, r = h["signals"], h["record"]
        print(f'{h["score"]:.4f}  [{r["kind"]:>10}]  {r["text"][:88]}')
        print(f'          vector={s["vector"]:.4f}  lexical={s["lexical"]:.4f}  '
              f'graph={s["graph"]:.4f}  recency={s["recency"]:.2f}')

def retrieve(query, **kw):
    return memory.recall(query, query_embedding=emb(query), **kw)

print("=== paraphrase, no shared keywords ===")
show(retrieve("how quickly do toggles propagate to users?", top_k=3))

print()
print("=== exact identifier ===")
show(retrieve("INC-2201", strategy="lexical", top_k=2))

In [ ]:
print("=== relational: what is connected to Meridian Labs? ===")
show(retrieve("Meridian Labs", strategy="relational", top_k=3))

## RAG: grounded generation

Retrieved memories become the LLM's context. Enter an OpenAI key to generate;
leave it blank and the cell prints the grounded context instead.

In [ ]:
import getpass, os
if not os.environ.get("OPENAI_API_KEY"):
    key = getpass.getpass("OpenAI API key (blank to skip generation): ")
    if key:
        os.environ["OPENAI_API_KEY"] = key

In [ ]:
def rag(question, top_k=4):
    hits = retrieve(question, top_k=top_k)
    context = "\n".join(f"[{i+1}] {h['record']['text']}" for i, h in enumerate(hits))
    if not os.environ.get("OPENAI_API_KEY"):
        print("No API key — grounded context memrust would hand the LLM:\n")
        print(context)
        return
    from openai import OpenAI
    resp = OpenAI().chat.completions.create(
        model="gpt-4o-mini",
        messages=[
            {"role": "system", "content":
             "Answer using ONLY the numbered memories provided. Cite like [1]. "
             "If the memories are insufficient, say so."},
            {"role": "user", "content": f"Memories:\n{context}\n\nQuestion: {question}"},
        ],
    )
    print(resp.choices[0].message.content)

rag("What happened with Orion and what should we do if it happens again?")

## Memory lifecycle

Memory that only grows is a liability. memrust ships a metabolism:

- **TTLs** — `working` memories expire and are durably swept
- **consolidation** — old episodic memories fold into semantic summaries
  (with `sources` provenance); we started the server with
  `--consolidate-after-secs 0` so it triggers immediately
- **snapshots** — export/restore a session's memory, id-preserving

In [ ]:
import time

# The collection's dimension was fixed at 384 by our first vector, so every
# remember passes an embedding from the same model — consistency is the rule.
memory.remember("scratch: comparing KV store configs right now",
                kind="working", ttl_seconds=3,
                embedding=emb("scratch: comparing KV store configs right now"))
print("live before expiry:", memory.health()["total_memories"])
time.sleep(4)
report = memory.run_lifecycle()
print("lifecycle:", report)
print("live after sweep:  ", memory.health()["total_memories"])

In [ ]:
# Consolidation: 4+ episodic memories in one session fold into one summary.
for i in range(4):
    note = f"standup note {i}: KV migration test batch {i} completed cleanly"
    memory.remember(note, kind="episodic", session_id="standup", embedding=emb(note))
report = memory.run_lifecycle()
print("consolidated batches:", report["batches_consolidated"])

summary_id = report["summaries"][0]
for h in memory.recall("KV migration progress",
                       query_embedding=emb("KV migration progress"), top_k=3):
    r = h["record"]
    if r["id"] == summary_id:
        print("summary:", r["text"][:120])
        print("distilled from", len(r["sources"]), "source memories")

In [ ]:
snap = memory.snapshot("kb")
print("snapshot:", len(snap["records"]), "records")
print("restore is idempotent — added:", memory.restore(snap["records"]))
memory.checkpoint()   # persist indexes, truncate the WAL
memory.health()

## Wrap-up

You built: local embeddings → vector + text + graph storage → hybrid explained
retrieval → grounded generation → lifecycle management. Next:
**03_pdf_rag_agents.ipynb** ingests a real PDF and wires memrust into a
LangGraph agent with the full multi-agent memory feature set.